# MLflow Example: Tracking Two Model Training Runs

In [1]:
!pip3 install scikit-learn


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [6]:
! pip3 install mlflow

  Using cached graphql_core-3.2.14-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.2.14-py3-none-any.whl (279 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.3.2
    Uninstalling graphql-core-3.3.2:
      Successfully uninstalled graphql-core-3.3.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gql 4.4.0 requires graphql-core<3.4,>=3.3.0a3, but you have graphql-core 3.2.14 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [8]:
!pip3 install dagshub

  Using cached graphql_core-3.3.2-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.3.2-py3-none-any.whl (366 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.2.14
    Uninstalling graphql-core-3.2.14:
      Successfully uninstalled graphql-core-3.2.14


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphene 3.4.3 requires graphql-core<3.3,>=3.1, but you have graphql-core 3.3.2 which is incompatible.
graphql-relay 3.2.0 requires graphql-core<3.3,>=3.2, but you have graphql-core 3.3.2 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='hugobouffier14', repo_name='dvc-project-hb', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=849230df-4fab-4a4b-86e5-54125a32ddc3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6a20552a38eedb465b2dcee7c9132bc0b07c3018f67c42ecd520a458d81c0daa




Output()

Accessing as hugobouffier14

Initialized MLflow to track repo "hugobouffier14/dvc-project-hb"

Repository hugobouffier14/dvc-project-hb initialized!

In [10]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [11]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


2026/11/16 17:24:23 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_V1.0.1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b39df757650f4b699541ebf0340c7d8b', creation_time=1794846265250, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794846265250, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [23]:
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            registered_model_name="digits-rf",
            serialization_format="cloudpickle",
        )
        print(f"{run_name}: accuracy={accuracy:.4f}")

In [24]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(
    n_estimators=20,
    max_depth=5,
    run_name="random_forest_model_run_1"
)

Training and Logging Model 1...


2026/11/16 17:33:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 17:33:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'digits-rf'.
2026/11/16 17:33:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 1
Created version '1' of model 'digits-rf'.


random_forest_model_run_1: accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/hugobouffier14/dvc-project-hb.mlflow/#/experiments/0/runs/5de46ec41ead408281ac8afef6f9a850
🧪 View experiment at: https://dagshub.com/hugobouffier14/dvc-project-hb.mlflow/#/experiments/0


In [25]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")


Training and Logging Model 2...


2026/11/16 17:34:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/16 17:34:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'digits-rf' already exists. Creating a new version of this model...
2026/11/16 17:34:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 2
Created version '2' of model 'digits-rf'.


random_forest_model_run_V1.0.2: accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/hugobouffier14/dvc-project-hb.mlflow/#/experiments/0/runs/8b9ba72b4fb242d2ae8d2323afe69cc2
🧪 View experiment at: https://dagshub.com/hugobouffier14/dvc-project-hb.mlflow/#/experiments/0


In [ ]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")